# Janome による日本語の形態素解析

## テキストファイルの準備

In [ ]:
# テキストファイルの読み込み

#   sangetsuki.txt は，https://wwws.kobe-c.ac.jp/~miura/Sc188/Data/Janome/sangetsuki.txt から
#   他に，
#   mojika.txt： https://wwws.kobe-c.ac.jp/~miura/Sc188/Data/Janome/mojika.txt
#   meijinden.txt： https://wwws.kobe-c.ac.jp/~miura/Sc188/Data/Janome/meijinden.txt
#   もあるので，試してみられたい

src = "sangetsuki.txt"              # 同じフォルダにあるテキストファイルの名前
f = open(src, encoding="utf_8_sig") # 文字コードが UTF BOM付き でない場合は，utf_8_sig を書き換える
text = f.read()
f.close()

print(text)

In [ ]:
# タイトル，著者，本文，書誌情報の分離

# 正規表現ライブラリをインポート
import re

title = text.split('\n')[0]             # 1行目をタイトルとみなす
author = text.split('\n')[1]            # 2行目を著者とみなす
body = re.split(r'\n\n\n+', text)[1]    # 2行以上の空行で3つに分けて，2つ目を本文とし，
info = re.split(r'\n\n\n+', text)[2]    # 3つ目を書誌情報とする

print(title, author)    # タイトルと著者の表示
print(body)             # 本文の表示

## 形態素解析

In [ ]:
# ライブラリから，必要な関数をインポート
from janome.tokenizer import Tokenizer
from janome.analyzer import Analyzer
from janome.tokenfilter import *

# 記号を取り除き，複合名詞は1つの名詞にまとめて，形態素解析を行う
tokenizer = Tokenizer()
token_filters = [CompoundNounFilter(), POSStopFilter('記号')]
a = Analyzer(tokenizer=tokenizer, token_filters=token_filters)

# 結果を表の形にして表示する
tab = [[t.surface] + t.part_of_speech.split(',') +
       [t.infl_type, t.infl_form, t.base_form, t.reading, t.phonetic]
       for t in a.analyze(body)]

import pandas as pd
tokens = pd.DataFrame(tab)
tokens.columns = [
    '表層形', '品詞', '品詞細分類1', '品詞細分類2', '品詞細分類3',
    '活用型', '活用形', '原形', '読み', '発音'
    ]
#tokens.to_csv('tokens.csv', index=False, encoding='utf_8_sig')

display(tokens)

# 日本語ワードクラウドの生成

In [ ]:
# 名詞, 動詞, 形容詞に限定して単語の(原形の)リストを作る
ind_filters = [CompoundNounFilter(), POSKeepFilter(['名詞', '動詞', '形容詞'])]
a2 = Analyzer(tokenizer=tokenizer, token_filters=ind_filters)
words_list = [ t.base_form for t in a2.analyze(text)]

print(words_list)

In [ ]:
# 一部の頻出語を除去する
rm_words = [
    "ある", "いる", "する", "せる", "なる", "これ", "それ", "あれ", "どれ",
    "この", "その", "あの", "どの", "もの", "こと", "よう", "れる", "られる",
    "の","ない"
    ]
for rw in rm_words:
    words_list = [w for w in words_list if w!=rw]

# 単語のリストを一つのテキスト(文字列)にまとめる
words = ' '.join(words_list)

display(words)

In [ ]:
# ワードクラウドの生成表示
from wordcloud import WordCloud
import matplotlib.pyplot as plt
wc = WordCloud(font_path="/c:/Windows/Fonts/YuGothM.ttc", width=500, height=500)
wc.generate(words)
plt.figure(dpi=150)     # dpi の数字を大きくすると，大きく表示
plt.imshow(wc)
plt.axis("off")         # 軸目盛りの非表示
plt.show()